# Falcon 9 mission-planning demo

This notebook compares three personas with different Falcon 9 destinations: the novice asks about a trip to the Moon, the experienced amateur asks about the ISS, and the expert specifies a 200 km LEO insertion. The novice's request stays plain-language and vague; the notebook adds clearly labeled payload/orbit defaults because the parser needs those values.

| Persona | What to notice |
| --- | --- |
| Novice | Moon-bound question, explicit demonstration defaults, accessible explanation, and glossary |
| Experienced amateur | ISS payload question plus separate model-rocket experience |
| Expert | Quantitative LEO screening case, margin and sensitivity analysis |
| Experienced amateur | Model-rocket details normalized to metric units and separated from Falcon 9 inputs |
| Expert | Stage parameters, budget margins, sensitivity, and model limitations |

The notebook calls the existing command-line pipeline and report generator; it does not change or replace their source code. Run it from the repository root. Mission parsing and the agent loop require the `TAMU_API_KEY` described in the README. Generated JSON and reports are saved under `reports/examples/`.

In [13]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "run_mission.py").is_file():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

ROCKET_SPEC = ROOT / "rockets" / "falcon9.txt"
EXAMPLES_DIR = ROOT / "examples"
REPORTS_DIR = ROOT / "reports" / "examples"
SCENARIOS = [
    ("novice", "Novice", EXAMPLES_DIR / "novice.txt"),
    ("experienced_amateur", "Experienced amateur", EXAMPLES_DIR / "experienced_amateur.txt"),
    ("expert", "Expert", EXAMPLES_DIR / "expert.txt"),
]
DEMO_DEFAULTS = {
    "novice": "\n\n[DEMO-ONLY DEFAULTS ADDED BY THIS NOTEBOOK]\nFor this comparison only, treat the 30 people as a 3,000 kg combined payload (100 kg per person), use a 200 km parking orbit, and interpret the destination as the Moon. Do not include a cabin, life support, or other hardware in the payload. These are explicit screening assumptions, not facts inferred by the parser.",
}

for slug, persona, path in SCENARIOS:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {persona} example input: {path}")
    print(f"{persona}: {path.relative_to(ROOT)}")

Novice: examples/novice.txt
Experienced amateur: examples/experienced_amateur.txt
Expert: examples/expert.txt


## 1. Compare the user inputs

Each text file contains the user's request and persona-specific context. The novice's saved input remains vague and conversational; the notebook adds labeled payload and parking-orbit defaults while preserving the Moon destination.

In [14]:
for slug, persona, path in SCENARIOS:
    print(f"\n{'=' * 16} {persona} {'=' * 16}\n")
    print(path.read_text(encoding="utf-8"))


================ Novice ================

# Persona: Novice — a curious student who watched a launch
[MISSION REQUEST]
I watched a rocket launch and got curious. Could a Falcon 9 take thirty people on a trip to the Moon? I don't know the rocket science words, and I might be imagining that the rocket just points at the Moon and flies there. Please explain what you think would happen in everyday language.

[PERSONA CONTEXT]
I have no rocketry background and don't know how much we need to know about the people or their trip. Please say what you assume about the trip and what you leave out, then add a short glossary for any technical terms in the report.


================ Experienced amateur ================

# Persona: Experienced amateur — a model-rocket builder learning mission planning
[MISSION REQUEST]
Could Falcon 9 carry a 3,000 kg payload to the International Space Station (ISS), into a 400 km parking orbit? I have launched model rockets for years, but haven't worked through orbi

## 2. Run the pipeline and generate reports

The notebook saves each mission result as JSON, then invokes the report generator with deterministic prose and benchmarking disabled. It adds a persona-specific interpretation to each generated Markdown report; all mission calculations remain the pipeline's results.

In [16]:
def run_command(args):
    completed = subprocess.run(args, cwd=ROOT, text=True, capture_output=True, check=False)
    if completed.returncode:
        detail = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(map(str, args))}\n{detail}")
    return completed.stdout


def fmt(value, digits=1):
    return "not reported" if value is None else f"{value:,.{digits}f}"


def sourced_value(stage, name):
    return (stage.get(name) or {}).get("value")


def append_notes(report_path, result, slug):
    agent = result.get("agent", {})
    facts = agent.get("facts", {})
    ascent = agent.get("ascent", {})
    transfer = agent.get("transfer", {})
    margin = agent.get("margin", {})
    requested = facts.get("requested_kg", result.get("request", {}).get("payload_kg"))
    destination = result.get("request", {}).get("destination", "the requested destination")
    ceiling = facts.get("ceiling_kg", agent.get("max_payload", {}).get("max_payload_kg"))

    if slug == "novice":
        lines = [
            "## Novice notes: assumptions and glossary",
            "",
            f"- **Payload assumption:** {fmt(requested, 0)} kg, using 100 kg per person for 30 people. Cabin, life support, supplies, and other mission hardware are not included.",
            "- **Destination assumption:** the Moon-bound request is modeled as a transfer from the default 200 km parking orbit. It does not model lunar capture, landing, or a return mission.",
            f"- **Plain-language result:** the model's payload ceiling is {fmt(ceiling, 0)} kg. Compare it with the requested payload above; this simplified budget check is not proof that people could safely fly.",
            "",
            "| Term | Plain-language meaning |",
            "| --- | --- |",
            "| Payload | The mass being carried, excluding rocket stages; cabin and life-support mass would also need to be counted. |",
            "| Parking orbit | A temporary orbit around Earth before the next major burn. |",
            "| TLI | Trans-lunar injection: a burn intended to put a spacecraft on a path toward the Moon. |",
            "| Delta-v | A measure of how much a vehicle can change its velocity; it is a budget, not simply its final speed. |",
            "| Apogee | The highest point of an orbit. This pipeline does not calculate an apogee for this mission. |",
            "",
            "The model does not establish human-rating, life-support adequacy, launch licensing, lunar capture, landing, or crew return.",
        ]
    elif slug == "experienced_amateur":
        twr = ascent.get("liftoff_twr")
        lines = [
            "## Experienced-amateur notes: normalized hobby context",
            "",
            "| Hobby-rocket detail | Supplied value | Normalized value | How it relates to this run |",
            "| --- | ---: | ---: | --- |",
            "| Motor | E-class | E-class | Describes the separate model rocket, not a Falcon 9 stage. |",
            f"| Pad mass | 4.5 lb | {4.5 * 0.45359237:.2f} kg | Not used in the Falcon 9 payload calculation. |",
            "| Static stability | 1.5 calibers | 1.5 calibers | Model-rocket stability measure; Falcon 9 aerodynamic/stability data are not supplied. |",
            f"| Launch-site elevation | 1,500 ft | {1500 * 0.3048:.1f} m | The mission model does not adjust losses for this site elevation. |",
            f"| Recovery goal | 0.5 mile | {0.5 * 1609.344:.1f} m | Parachute recovery is outside the orbital mission model. |",
            "",
            f"The Falcon 9 model reports liftoff thrust-to-weight of **{fmt(twr, 2)}** for this payload; that is not the model rocket's thrust-to-weight ratio. A hobby-rocket apogee cannot be estimated from motor class and pad mass alone. It would require a motor thrust curve and burn duration, vehicle drag/geometry, and flight conditions.",
            "",
            "Hobby intuition remains useful for asking about stability, thrust margin, and recovery, but orbital energy and staged-mass budgets dominate this mission. None of the hobby-rocket values above are treated as Falcon 9 data.",
        ]
    elif slug == "expert":
        from src.ascent_loop.physics import build_rocket, find_max_payload

        rows = []
        rocket_spec = {"name": result.get("rocket", {}).get("name", "Falcon 9"), "stages": []}
        for index, stage in enumerate(result.get("rocket", {}).get("stages", []), start=1):
            dry = sourced_value(stage, "dry_mass_kg")
            prop = sourced_value(stage, "propellant_mass_kg")
            isp_vac = sourced_value(stage, "isp_vac_s")
            isp_sl = sourced_value(stage, "isp_sl_s")
            wet_mass = (dry or 0) + (prop or 0)
            prop_fraction = prop / wet_mass * 100 if wet_mass else None
            rows.append(f"| {index} | {fmt(dry, 0)} | {fmt(prop, 0)} | {fmt(isp_sl, 0)} | {fmt(isp_vac, 0)} | {fmt(prop_fraction, 1)}% |")
            rocket_spec["stages"].append({
                "dry_mass_kg": dry,
                "propellant_mass_kg": prop,
                "isp_vac_s": isp_vac,
                "isp_sl_s": isp_sl,
                "thrust_sl_n": sourced_value(stage, "thrust_sl_n"),
            })
        reserve_dv = margin.get("reserved_transfer_dv_ms", transfer.get("reserved_transfer_dv_ms", transfer.get("dv_total_ms", 0)))
        reserve_fraction = margin.get("reserve_fraction")
        combined_margin = margin.get("margin_ms")
        rocket = build_rocket(rocket_spec)
        sensitivity = []
        for loss_budget in (1000.0, 1500.0, 2000.0):
            ceiling_at_loss = find_max_payload(rocket, losses_ms=loss_budget, required_extra_dv_ms=reserve_dv)["max_payload_kg"]
            sensitivity.append(f"| {loss_budget:,.0f} | {ceiling_at_loss:,.0f} |")
        ascent_reserve_dv = ascent.get("dv_needed_ms", 0) * 0.05
        ceiling_plus_reserve = find_max_payload(rocket, losses_ms=1500.0, required_extra_dv_ms=reserve_dv + ascent_reserve_dv)["max_payload_kg"]
        lines = [
            "## Expert screening annex",
            "",
            "### Vehicle parameters",
            "",
            "| Stage | Dry mass (kg) | Propellant (kg) | Isp, SL (s) | Isp, vacuum (s) | Propellant / wet stage mass |",
            "| ---: | ---: | ---: | ---: | ---: | ---: |",
            *rows,
            "",
            f"Requested payload: **{fmt(requested, 0)} kg to {destination}**; liftoff T/W: **{fmt(ascent.get('liftoff_twr'), 2)}**. The report's burn table contains per-stage mass states and delta-v. Ascent delta-v: **{fmt(ascent.get('dv_total_ms'))} m/s available** versus **{fmt(ascent.get('dv_needed_ms'))} m/s required**; transfer delta-v: **{fmt(transfer.get('dv_total_ms'))} m/s**; applied transfer reserve: **{fmt(reserve_dv)} m/s ({fmt((reserve_fraction or 0) * 100, 1)}%)**; residual combined margin: **{fmt(combined_margin)} m/s**; mission payload ceiling: **{fmt(ceiling, 0)} kg**.",
            "",
            "### Sensitivity: fixed loss budget, all else held constant",
            "",
            "| Ascent-loss budget (m/s) | Mission payload ceiling (kg) |",
            "| ---: | ---: |",
            *sensitivity,
            "",
            f"The pipeline applies a {fmt((reserve_fraction or 0) * 100, 1)}% reserve to modeled transfer delta-v; it does not apply the requested 5% ascent reserve. A separate sensitivity case adding 5% of the required ascent delta-v, on top of the modeled transfer reserve, gives a payload ceiling of **{ceiling_plus_reserve:,.0f} kg**.",
            "",
            "### Failure modes and limitations",
            "",
            "| Case | Treatment |",
            "| --- | --- |",
            "| Ascent/transfer budget shortfall | Quantitatively screened by the saved budget and margin outputs. |",
            "| Low liftoff thrust-to-weight | Nominal check reported; engine degradation/transients not simulated. |",
            "| Engine-out, stage separation, or guidance failure | Not modeled; no fault tree or reliability probabilities are available. |",
            f"| Destination-specific operations | {('ISS rendezvous/docking and plane changes' if destination == 'ISS' else 'lunar capture, landing, and return') if destination in {'ISS', 'Moon', 'TLI'} else 'orbit insertion details beyond the simplified budget'} are outside scope. |",
            "",
            "The deterministic sensitivity sweep varies only the fixed ascent-loss budget while holding vehicle data and transfer reserve constant. Nominal sequential staging is assumed. These results are not uncertainty bounds or a flight-design release.",
        ]
    else:
        raise ValueError(f"Unknown persona: {slug}")

    with report_path.open("a", encoding="utf-8") as report_file:
        report_file.write("\n\n" + "\n".join(lines) + "\n")


results = {}
for slug, persona, input_path in SCENARIOS:
    output_dir = REPORTS_DIR / slug
    output_dir.mkdir(parents=True, exist_ok=True)
    result_path = output_dir / "result.json"
    report_path = output_dir / "report.md"
    request = input_path.read_text(encoding="utf-8") + DEMO_DEFAULTS.get(slug, "")
    raw_json = run_command([sys.executable, "-m", "src.run_mission", request, str(ROCKET_SPEC), "5"])
    result = json.loads(raw_json)
    result_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    run_command([sys.executable, "-m", "src.report", str(result_path), "--out", str(output_dir)])
    append_notes(report_path, result, slug)
    results[slug] = result
    print(f"{persona}: {report_path.relative_to(ROOT)} (JSON: {result_path.relative_to(ROOT)})")

Novice: reports/examples/novice/report.md (JSON: reports/examples/novice/result.json)
Experienced amateur: reports/examples/experienced_amateur/report.md (JSON: reports/examples/experienced_amateur/result.json)
Expert: reports/examples/expert/report.md (JSON: reports/examples/expert/result.json)


## 3. Review the generated reports

Each report contains the pipeline verdict, burn/budget details, assumptions and provenance, and persona-specific interpretation. Inspect those assumptions before interpreting a feasibility result. The estimate is **not** a flight plan or a claim that the vehicle can safely transport people.

In [15]:
for slug, persona, _ in SCENARIOS:
    report_path = REPORTS_DIR / slug / "report.md"
    print(f"{persona}: {report_path.relative_to(ROOT)} — {report_path.stat().st_size:,} bytes")

Novice: reports/examples/novice/report.md — 4,765 bytes
Experienced amateur: reports/examples/experienced_amateur/report.md — 4,648 bytes
Expert: reports/examples/expert/report.md — 5,215 bytes
